In [1]:
import pandas as pd
import numpy as np

In [2]:
# -- Nombramos cada dataset --

df_1 = pd.read_excel("NPE_bruto_3_limpio.xlsx")
df_2 = pd.read_excel("responses_1y2_limpio.xlsx")

In [3]:
# -- Observamos sus caracteristicas generales --

print("DF1 shape:", df_1.shape)
print("DF2 shape:", df_2.shape)

print("\nColumnas DF1:")
print(df_1.columns.tolist())

print("\nColumnas DF2:")
print(df_2.columns.tolist())

DF1 shape: (1192, 44)
DF2 shape: (3378, 52)

Columnas DF1:
['nombre_completo', 'fecha_nacimiento', 'cuerpo_seguridad', 'descripcion_discapacidad', 'discapacidad_visual', 'problema_coordinacion', 'falta_fuerza', 'otro', 'extremidad', 'practica_deporte', 'deporte_actual', 'tiempo_deporte_semana', 'clasificado_panel', 'entrenador_actual', 'interes_competir_paralimpico', 'atletismo', 'badminton', 'baloncesto_silla_ruedas', 'boccia', 'ciclismo', 'escalada', 'esgrima_silla_ruedas', 'futbol_ciegos', 'goalball', 'powerlifting', 'hipica', 'judo', 'natacion', 'piraguismo', 'remo', 'rugby_silla_ruedas', 'taekwondo', 'tenis_mesa', 'tenis_silla_ruedas', 'tiro_arco', 'tiro_olimpico', 'triatlon', 'voleibol_sentado', 'biatlon', 'curling_silla_ruedas', 'esqui_alpino', 'esqui_fondo', 'hockey_hielo', 'snowboard']

Columnas DF2:
['nombre_completo', 'fecha_nacimiento', 'cuerpo_seguridad', 'lesion_medular', 'falta_fuerza', 'rango_movimiento', 'extremidad', 'problema_coordinacion', 'baja_estatura', 'discapac

In [4]:
# -- Vemos diferencias de columnas --

cols_df1 = set(df_1.columns)
cols_df2 = set(df_2.columns)

print("Solo en DF1:")
print(cols_df1 - cols_df2)

print("\nSolo en DF2:")
print(cols_df2 - cols_df1)

Solo en DF1:
set()

Solo en DF2:
{'discapacidad_intelectual', 'num_extremidades_amputadas', 'lesion_medular', 'nivel_discapacidad_visual', 'rango_movimiento', 'miembro_superior', 'miembro_inferior', 'baja_estatura'}


In [5]:
# -- Vemos tipos de datos --

print(df_1.dtypes)
print("\n")
print(df_2.dtypes)

nombre_completo                  object
fecha_nacimiento                 object
cuerpo_seguridad                float64
descripcion_discapacidad         object
discapacidad_visual             float64
problema_coordinacion           float64
falta_fuerza                    float64
otro                              int64
extremidad                        int64
practica_deporte                float64
deporte_actual                   object
tiempo_deporte_semana           float64
clasificado_panel                 int64
entrenador_actual               float64
interes_competir_paralimpico    float64
atletismo                       float64
badminton                       float64
baloncesto_silla_ruedas         float64
boccia                          float64
ciclismo                        float64
escalada                        float64
esgrima_silla_ruedas            float64
futbol_ciegos                   float64
goalball                        float64
powerlifting                    float64


In [6]:
# -- Unificamos las fechas de nacimiento --
# Las vacías se mantienen como null (NaT)

df_1["fecha_nacimiento"] = pd.to_datetime(df_1["fecha_nacimiento"], errors="coerce")
df_2["fecha_nacimiento"] = pd.to_datetime(df_2["fecha_nacimiento"], errors="coerce")

In [7]:
# -- Se crea columna 'edad' --

hoy = pd.Timestamp.today()

df_1["edad"] = ((hoy - df_1["fecha_nacimiento"]).dt.days / 365.25).round(0)
df_2["edad"] = ((hoy - df_2["fecha_nacimiento"]).dt.days / 365.25).round(0)

df_1.drop(columns=["fecha_nacimiento"], inplace=True)
df_2.drop(columns=["fecha_nacimiento"], inplace=True)

df_1["edad"] = df_1["edad"].astype("Int64")
df_2["edad"] = df_2["edad"].astype("Int64")

In [8]:
# -- Unificamos cuerpo_seguridad --
# df_1 ya viene numérico; df_2 hay filas como texto

df_2["cuerpo_seguridad"] = df_2["cuerpo_seguridad"].replace({
    "No": 0,
    "Sí": 1,
    "Si": 1
})

df_1["cuerpo_seguridad"] = pd.to_numeric(df_1["cuerpo_seguridad"], errors="coerce")
df_2["cuerpo_seguridad"] = pd.to_numeric(df_2["cuerpo_seguridad"], errors="coerce")

/var/folders/8p/pfbvqt715kl1tm3v1v3bx7qm0000gq/T/ipykernel_1679/2546548784.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_2["cuerpo_seguridad"] = df_2["cuerpo_seguridad"].replace({


In [9]:
# -- En df_2 usamos 'nivel_discapacidad_visual' para reconstruir 'discapacidad_visual'
# Regla:
# 0 -> 0
# 1 o 2 -> 1
# Solo queda 'discapacidad_visual'

df_2["nivel_discapacidad_visual"] = pd.to_numeric(df_2["nivel_discapacidad_visual"], errors="coerce").fillna(0)

df_2["discapacidad_visual"] = df_2["nivel_discapacidad_visual"].map({
    0: 0,
    1: 1,
    2: 1
}).fillna(0)

df_2.drop(columns=["nivel_discapacidad_visual"], inplace=True)

In [10]:
# -- En df_2 fusionamos 'discapacidad_intelectual' con 'otro'
# Si cualquiera de las dos es 1, 'otro' será 1
# Solo queda la columna 'otro'

df_2["discapacidad_intelectual"] = pd.to_numeric(df_2["discapacidad_intelectual"], errors="coerce").fillna(0)
df_2["otro"] = pd.to_numeric(df_2["otro"], errors="coerce").fillna(0)

df_2["otro"] = (
    (df_2["otro"] == 1) |
    (df_2["discapacidad_intelectual"] == 1)
).astype(int)

df_2.drop(columns=["discapacidad_intelectual"], inplace=True)

In [11]:
# -- Agregamos columnas extras en df_1 --
# Conservar columnas del df_2:
# lesion_medular, rango_movimiento, baja_estatura
# Si df_1 no tiene, se crean vacías

for col in ["lesion_medular", "rango_movimiento", "baja_estatura"]:
    if col not in df_1.columns:
        df_1[col] = np.nan

In [12]:
# -- Aseguramos tipo númerico en columnas binarias --

cols_numericas = [
    "cuerpo_seguridad",
    "discapacidad_visual",
    "problema_coordinacion",
    "falta_fuerza",
    "extremidad",
    "lesion_medular",
    "rango_movimiento",
    "baja_estatura",
    "otro",
    "practica_deporte",
    "tiempo_deporte_semana",
    "clasificado_panel",
    "entrenador_actual",
    "interes_competir_paralimpico"
]

for col in cols_numericas:
    if col in df_1.columns:
        df_1[col] = pd.to_numeric(df_1[col], errors="coerce")
    if col in df_2.columns:
        df_2[col] = pd.to_numeric(df_2[col], errors="coerce")

In [13]:
# -- Aseguramos tipo númerico en deportes --

cols_deportes = [
    "atletismo", "badminton", "baloncesto_silla_ruedas", "boccia", "ciclismo",
    "escalada", "esgrima_silla_ruedas", "futbol_ciegos", "goalball", "powerlifting",
    "hipica", "judo", "natacion", "piraguismo", "remo", "rugby_silla_ruedas",
    "taekwondo", "tenis_mesa", "tenis_silla_ruedas", "tiro_arco", "tiro_olimpico",
    "triatlon", "voleibol_sentado", "biatlon", "curling_silla_ruedas",
    "esqui_alpino", "esqui_fondo", "hockey_hielo", "snowboard"
]

for col in cols_deportes:
    if col in df_1.columns:
        df_1[col] = pd.to_numeric(df_1[col], errors="coerce")
    if col in df_2.columns:
        df_2[col] = pd.to_numeric(df_2[col], errors="coerce")

In [14]:
# -- Biniarizamos todos los deportes --

cols_deportes = [
'atletismo','badminton','baloncesto_silla_ruedas','boccia','ciclismo',
'escalada','esgrima_silla_ruedas','futbol_ciegos','goalball','powerlifting',
'hipica','judo','natacion','piraguismo','remo','rugby_silla_ruedas',
'taekwondo','tenis_mesa','tenis_silla_ruedas','tiro_arco','tiro_olimpico',
'triatlon','voleibol_sentado','biatlon','curling_silla_ruedas',
'esqui_alpino','esqui_fondo','hockey_hielo','snowboard'
]

# Convertir a binario: 0 se mantiene, cualquier otro valor -> 1
for col in cols_deportes:
    df_1[col] = (df_1[col].fillna(0) > 0).astype(int)
    df_2[col] = (df_2[col].fillna(0) > 0).astype(int)

for col in cols_deportes:
    print(col, sorted(df_1[col].unique()))

atletismo [np.int64(0), np.int64(1)]
badminton [np.int64(0), np.int64(1)]
baloncesto_silla_ruedas [np.int64(0), np.int64(1)]
boccia [np.int64(0), np.int64(1)]
ciclismo [np.int64(0), np.int64(1)]
escalada [np.int64(0), np.int64(1)]
esgrima_silla_ruedas [np.int64(0), np.int64(1)]
futbol_ciegos [np.int64(0), np.int64(1)]
goalball [np.int64(0), np.int64(1)]
powerlifting [np.int64(0)]
hipica [np.int64(0), np.int64(1)]
judo [np.int64(0), np.int64(1)]
natacion [np.int64(0), np.int64(1)]
piraguismo [np.int64(0), np.int64(1)]
remo [np.int64(0), np.int64(1)]
rugby_silla_ruedas [np.int64(0), np.int64(1)]
taekwondo [np.int64(0), np.int64(1)]
tenis_mesa [np.int64(0), np.int64(1)]
tenis_silla_ruedas [np.int64(0), np.int64(1)]
tiro_arco [np.int64(0), np.int64(1)]
tiro_olimpico [np.int64(0), np.int64(1)]
triatlon [np.int64(0), np.int64(1)]
voleibol_sentado [np.int64(0), np.int64(1)]
biatlon [np.int64(0), np.int64(1)]
curling_silla_ruedas [np.int64(0), np.int64(1)]
esqui_alpino [np.int64(0), np.int64(1

In [15]:
# -- Orden final de columnas --

columnas_finales = [
    "nombre_completo",
    "edad",
    "cuerpo_seguridad",
    "descripcion_discapacidad",
    "discapacidad_visual",
    "problema_coordinacion",
    "falta_fuerza",
    "extremidad",
    "lesion_medular",
    "rango_movimiento",
    "baja_estatura",
    "otro",
    "practica_deporte",
    "deporte_actual",
    "tiempo_deporte_semana",
    "clasificado_panel",
    "entrenador_actual",
    "interes_competir_paralimpico",
    "atletismo",
    "badminton",
    "baloncesto_silla_ruedas",
    "boccia",
    "ciclismo",
    "escalada",
    "esgrima_silla_ruedas",
    "futbol_ciegos",
    "goalball",
    "powerlifting",
    "hipica",
    "judo",
    "natacion",
    "piraguismo",
    "remo",
    "rugby_silla_ruedas",
    "taekwondo",
    "tenis_mesa",
    "tenis_silla_ruedas",
    "tiro_arco",
    "tiro_olimpico",
    "triatlon",
    "voleibol_sentado",
    "biatlon",
    "curling_silla_ruedas",
    "esqui_alpino",
    "esqui_fondo",
    "hockey_hielo",
    "snowboard"
]

In [16]:
# -- Mantener el mismo esquema en los dataset --
# Si falta alguna columna, se crea vacía

for col in columnas_finales:
    if col not in df_1.columns:
        df_1[col] = np.nan
    if col not in df_2.columns:
        df_2[col] = np.nan

df_1 = df_1[columnas_finales]
df_2 = df_2[columnas_finales]

In [17]:
# -- Revisión final de datasets --

print("DF1 shape:", df_1.shape)
print("DF2 shape:", df_2.shape)
print("Mismas columnas:", list(df_1.columns) == list(df_2.columns))
print(df_1.dtypes)
print(df_2.dtypes)

DF1 shape: (1192, 47)
DF2 shape: (3378, 47)
Mismas columnas: True
nombre_completo                  object
edad                              Int64
cuerpo_seguridad                float64
descripcion_discapacidad         object
discapacidad_visual             float64
problema_coordinacion           float64
falta_fuerza                    float64
extremidad                        int64
lesion_medular                  float64
rango_movimiento                float64
baja_estatura                   float64
otro                              int64
practica_deporte                float64
deporte_actual                   object
tiempo_deporte_semana           float64
clasificado_panel                 int64
entrenador_actual               float64
interes_competir_paralimpico    float64
atletismo                         int64
badminton                         int64
baloncesto_silla_ruedas           int64
boccia                            int64
ciclismo                          int64
escalada      

In [18]:
# -- Fusionamos ambos datasets --
df_final = pd.concat([df_1, df_2], ignore_index=True)

# -- Resultado Final y guardado --
df_final.to_excel("Dataset_fusionado_limpio.xlsx", index=False)

print("Archivo generado: Dataset_fusionado_limpio.xlsx")

Archivo generado: Dataset_fusionado_limpio.xlsx
